In [1]:
import geopandas as gpd
import numpy as np
from pathlib import Path

import mnk.substrat as subkart
import mnk
import pandas as pd


## Create features tiffs

In [2]:
res = subkart.features.RESOLUTION
nodata = 255
crs = "EPSG:25833"

In [3]:
bolge = mnk.sources.bolge_exposure()
dem_norge = mnk.sources.dem_data()

In [4]:
gdf_sea_map = mnk.sources.sea_map_basisdata()
crs = gdf_sea_map.crs
gdf_sea_map = subkart.features.depth_preprocess(gdf_sea_map)
transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map, res=res)

In [5]:
dem = mnk.sources.dem_data()

In [6]:
dem = dem.crop(bounds)
dem = subkart.utils.resample_dem(dem, out_shape, transform, crs)

In [7]:
bolge = mnk.sources.bolge_exposure()

In [8]:
bolge = bolge.reproject(
    crs=crs, res=res,
    bounds=dict(left=bounds[0], bottom=bounds[1], right=bounds[2], top=bounds[3])
)

In [9]:
gdf_points = mnk.sources.depth_point_data()
print(f"Depth points: {len(gdf_points):,}")


Depth points: 6,047,001


In [10]:
X, valid, out_shape, transform = subkart.features.build(
    dem, gdf_sea_map, bolge, valid_mask=None, res=res, dtype=np.float32,
    gdf_points=gdf_points,
)

Computing interpolated depth raster...


  Point-refined 120 wide-range polygons.


Preparing sea_avg_depth...


Preparing sea_avg_slope...


/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


...

/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_convexity...


/home/jovyan/marint-naturkart-niva/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing wave exposure...


Stacking feature arrays...


In [11]:
subkart.utils.save_feature_rasters(Path('/home/jovyan/shared/common/KIM/features'), 'norge', X, valid, transform, out_shape, dem.nodata)